В Postgres есть три основных типа данных для работы со строками: **character**, **character varying** и **text**.

## character

Cтрока фиксированной длины, дополненная пробелами.

Длина строки такого типа всегда одинакова и задаётся в скобках.

Например, в столбце character(5) всегда будет пять символов: строку большей длины туда вставить не получится, а строка меньшей длины будет дополняться ведущими пробелами. Слово "SQL" в таком столбце будет выглядеть как "  SQL".

Основной паттерн использования такого типа — универсальные справочники буквенных кодов, например код страны в стандарте ISO (RU, US, UK и т. д.).

## character varying

Строка ограниченной переменной длины.

Например, в столбце типа character varying(5) нельзя будет хранить строку большей длины, но могут быть любые строки с меньшей длиной.

Этот тип данных повсеместно используется для хранения данных, поскольку позволяет ограничить ввод, сохраняя при этом возможность иметь строки произвольной длины.

## text

Cтрока неограниченной длины.

Самый удобный тип для пользователя, но самый тяжеловесный для администратора баз данных, так как в строку можно записать любой текст.

Для удобства все текстовые поля в нашем датасете с доставками представлены типом text.

## <center> Соединение строк

Для начала познакомимся с оператором конкатенации строк — || (две вертикальные черты). Он позволяет объединять две и более строки.

Конструкции с оператором соединения строк записываются следующим образом:
```sql
строка1 || строка2 || ... || строкаN

Важно! Результатом соединения любых типов строковых данных будет тип text.

>Напишем запрос, который позволит подготовить простые select-запросы для всех таблиц из схемы.
```sql
select 'select * from '||t.table_schema||'.'||t.table_name||';' query
from information_schema.tables t
where table_schema = 'shipping'

В результате должно получиться пять SQL-запросов, по одному к каждой таблице из схемы shipping. 

Как мы видим, соединять можно и рукописный текст, и значения столбцов в любом произвольном порядке.

>Важно! Если вы соединяете любую строку и NULL, то результатом будет NULL. Поэтому, если вы формируете какой-то текст на основе поля, в котором присутствует NULL, используйте оператор ```coalesce```.

>Составим текстовый шаблон сообщения о доставке по конкретному водителю для наших клиентов. Напишите SQL-запрос, который выведет следующее сообщение для каждого водителя по форме:

Ваш заказ доставит водитель #Имя Фамилия#. Его контактный номер: #Номер#
Где #Имя Фамилия# и #Номер# взяты из справочника водителей. Если номер не указан, то выведите прочерк (-). Для номеров рекомендуем использовать COALESCE. Пример из таблицы для наглядности:

Ваш заказ доставит водитель Adel Al-Alawi. Его контактный номер: (901) 947-4433
Столбец к выдаче — msg (текст сообщения).

```sql
select 'Ваш заказ доставит водитель ' || d.first_name || ' ' || d.last_name || '. Его контактный номер: ' || d.phone msg
from sql.driver d
where d.phone is not null
union
select coalesce('Ваш заказ доставит водитель ' || d.first_name || ' ' || d.last_name || '. Его контактный номер: ' || '-')  
from sql.driver d
where d.phone is null

## <center> Функции UPPER() и LOWER()

Функции upper(your_text) и lower(your_text) переводят каждый символ вашего текста в верхний и нижний регистр соответственно.

>Пример:
```sql
select upper('Abc') s1 ,lower('xYz') s2

Чаще всего эти функции используются для унификации и стандартизации, особенно они актуальны для данных, введённых вручную.

Например, названия города в анкете можно написать разными способами, но символьный состав останется одним и тем же (Москва, москва, МОСКВА).

Результат функций upper() и lower() — тоже строковый, а значит, к нему можно применять все функции, применимые к этому типу данных.

>Cоставим справочник названий клиентов, у которых более десяти доставок. Данные сохраним в нижнем регистре, чтобы передавать их в другие системы (например, для обзвона), которые не чувствительны к регистру. Напишите запрос, который выводит все id названий клиентов, у которых более десяти доставок, в нижнем регистре. Отсортируйте результат по cust_id в порядке возрастания. Столбцы в выдаче: cust_id (id клиента) и cust_name (название клиента в нижнем регистре).
```sql
select 
    c.cust_id,
    lower(c.cust_name) cust_name
from 
    sql.customer c
join sql.shipment s on c.cust_id = s.cust_id
group by c.cust_id
having count(s.ship_id) > 10
order by 1

## <center> Replace()

С помощью функции replace() можно заменять символы в строках.

Запись строится следующим образом:
```sql
replace(string text, from text, to text)

Эта запись означает, что в исходной строке string мы заменяем все вхождения строки from на строку to.

```sql
select replace('малако','а','о')

>Сделаем из слова «машина» слово «матрас»
```sql
select replace('машина','шина','трас')

Если вы хотите удалить из строки какие-то символы, то третьим параметром (to) передайте пустую строку ''(одинарные кавычки без символа внутри).

>Например, сделаем из строки "Hello, world!" строку "Hello!".
```sql
select replace('Hello, world!',', world','')

>Составим справочник utm-меток, для того чтобы передавать город и штат прямо в адресной строке. (Если вы не знаете, что такое utm-метка, почитайте статью на Вики. К программе курса это не относится, но знать полезно.) Напишите SQL-запрос, который выведет список сочетаний из справочника следующего вида: название_штата__название_города, где названия штата и города взяты из справочника городов и переведены в нижний регистр. Столбец к выдаче — utm (форматированный штат-город). Отсортируйте полученный справочник по алфавиту. Обратите внимание! Все пробелы в названиях городов и штатов замените символом '_' (одно нижнее подчёркивание), а для разделения названий города и штата используйте '__' (два последовательных нижних подчёркивания). Пример из таблицы для наглядности: new_jersey__union_city
```sql
select
replace(replace(lower(c.state),' ','_') || '$' || replace(lower(c.city_name),' ','_'),'$','__') utm
from 
    sql.city c
order by 1

## <center> Left() и Right()

Функции left(string,n) и right(string,n) отрезают n левых или правых символов от строки, поданной на вход. Давайте разобьём строку 'Один два три' на слова, используя эти функции.
```sql
with t as
(
select 'Один два три'::text sample_string
)
select 
 left(t.sample_string,4) one, /*берём 4 левых символа строки*/
 right(left(t.sample_string,8),3) two, /*берём 8 левых символов строки, потом 3 правых от результата*/
 right(t.sample_string,3) three /*берём 3 правых символа от строки*/
from t

```sql
select left('0123456789', - 2), right('0123456789', - 2)

>Представим, что к вам пришёл разработчик, который хочет сократить поле state в таблице city до четырёх символов, и попросил проверить, останeтся ли значения в нём уникальными. Чтобы ответить на этот вопрос, напишите SQL-запрос, который выведет первые четыре символа названия штата и количество уникальных названий штатов, которому они соответствуют. Оставьте только те, которые относятся к двум и более штатам. Добавьте сортировку по первому столбцу. Столбцы в выдаче: code (четыре первых символа в названии штата), qty (количество уникальных названий штата, начинающихся с этих символов).
```sql
select 
left(c.state, 4) code,
count(distinct c.state) qty
from sql.city c
group by left(c.state, 4)
having count(distinct c.state) >= 2
order by 1

## <center> Format()

Функция format() используется для составления форматированного текста с подстановками. То же самое можно сделать через конкатенацию строк, но это неудобно и громоздко.

Синтаксис функции выглядит следующим образом:
```sql
format(formatstr text [, argument1 text,argument2 text...])

где formatstr — это шаблон, который мы передаём. Это обычная строка, в которой указаны места для подстановки аргумента.

```sql
select format('Hello, %s!', d.first_name) from shipping.driver d

Комбинация символов %s обозначает, что вместо них будет подставлен один из аргументов, причём в том же порядке, что и в исходном столбце.

>Напишем запрос, который описывает содержимое каждой строки в таблице в виде текста.
```sql
select format('driver_id = %s, first_name = %s, last_name = %s, address = %s, zip_code = %s, phone = %s, city_id = %s', driver_id, first_name, last_name, address, zip_code, phone, city_id) from shipping.driver d

Мы перечислили в строке семь пропусков (плэйсхолдеров, или мест для подстановки, — %s), передали семь параметров (все столбцы таблицы) и получили шаблон, заполненный значениями для каждой строки.

Если в вашем шаблоне присутствует одинарная кавычка, то для удобства можно вместо одинарных кавычек использовать $$ (два знака доллара):
```sql
select $$ some_string with quotes ' $$

>Давайте подготовим географическую сводку для каждого города. Напишите SQL-запрос, который выведет описание региона в следующем формате:


[city_name] is located in [state]. There's [population] people living there. Its area is [area]

Обратите внимание, точку в конце ставить не нужно. Отсортируйте по названию города в алфавитном порядке. Столбец к выдаче — str (сводка). Пример:


Abilene is located in Texas. There's 115930 people living there. Its area is 105.
```sql
select
format($$ %s is located in %s. There's %s people living there. Its area is %s $$, c.city_name, c.state, c.population, c.area) str
from sql.city c
order by 1